In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Import pipeline/fungsi dari app Anda
import app as detector


# =========================
# KONFIGURASI
# =========================
TEST_CSV_PATH = "DataFiles/dataset_test.csv"
URL_COL = "url"
LABEL_COL = "status"

# Label asli di dataset_test.csv:
# 0 = phishing, 1 = benign
DATA_PHISHING_LABEL = 0
DATA_BENIGN_LABEL = 1

# Label model (sesuai training/app.py):
# 1 = phishing, 0 = benign
MODEL_PHISHING_LABEL = 1
MODEL_BENIGN_LABEL = 0

# Jika True: hanya ekstrak fitur URL (lebih cepat, tidak fetch HTML)
# Jika False: ikut fitur web-content bila dibutuhkan model
FORCE_URL_ONLY = False

# Progress report per N data
PROGRESS_EVERY = 20_000


# =========================
# LOAD DATA UJI
# =========================
df = pd.read_csv(TEST_CSV_PATH)

missing_cols = [c for c in [URL_COL, LABEL_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan di data uji: {missing_cols}")

df[URL_COL] = df[URL_COL].astype(str).str.strip()
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df[df[URL_COL] != ""].copy()
df = df[df[LABEL_COL].isin([DATA_PHISHING_LABEL, DATA_BENIGN_LABEL])].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

if df.empty:
    raise ValueError("Data uji kosong setelah cleaning.")

total_n = len(df)
print(f"Total data uji valid: {total_n:,}")


# =========================
# LOAD ARTIFACT MODEL
# =========================
cols = detector.get_feature_columns()
medians = detector.get_feature_medians()
rf_model, xgb_model, meta_model = detector.load_models()

if not cols:
    raise ValueError("Feature columns kosong. Pastikan artifact fitur tersedia.")

include_web_content = detector._resolve_include_web_content(cols) and (not FORCE_URL_ONLY)
print(f"Jumlah fitur model: {len(cols)}")
print(f"Web content dipakai: {include_web_content}")
print(f"Meta model tersedia: {meta_model is not None}")


# =========================
# HELPER
# =========================
def proba_for_label(model, X, target_label):
    classes = list(getattr(model, "classes_", []))
    try:
        probs = model.predict_proba(X)[0]
        if target_label in classes:
            idx = classes.index(target_label)
            return float(probs[idx])

        # fallback binary
        if len(probs) >= 2:
            return float(probs[1]) if target_label == 1 else float(probs[0])
        return float(probs[0])
    except Exception:
        pred = int(model.predict(X)[0])
        return 1.0 if pred == target_label else 0.0


def convert_data_label_to_model_label(data_label):
    return MODEL_PHISHING_LABEL if int(data_label) == DATA_PHISHING_LABEL else MODEL_BENIGN_LABEL


def clamp01(x):
    return float(min(max(float(x), 0.0), 1.0))


def evaluate_and_show(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[MODEL_PHISHING_LABEL, MODEL_BENIGN_LABEL])

    cm_df = pd.DataFrame(
        cm,
        index=["Actual Phishing (model=1)", "Actual Benign (model=0)"],
        columns=["Pred Phishing (model=1)", "Pred Benign (model=0)"],
    )

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=MODEL_PHISHING_LABEL, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=MODEL_PHISHING_LABEL, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=MODEL_PHISHING_LABEL, zero_division=0)

    print("\n" + "=" * 72)
    print(name)
    print("=" * 72)
    display(cm_df)
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision (phishing=1 model): {prec:.4f}")
    print(f"Recall    (phishing=1 model): {rec:.4f}")
    print(f"F1-score  (phishing=1 model): {f1:.4f}")


def bucket_summary(model_name, probs, split=0.6):
    s = pd.Series(probs, dtype=float)
    b_low = s[(s >= 0.0) & (s < split)]
    b_high = s[(s >= split) & (s <= 1.0)]

    rows = [
        {
            "model": model_name,
            "bucket": f"0.0-<{split}",
            "count": int(b_low.shape[0]),
            "pct": (b_low.shape[0] / len(s) * 100.0) if len(s) else 0.0,
            "avg_prob": float(b_low.mean()) if len(b_low) else np.nan,
        },
        {
            "model": model_name,
            "bucket": f"{split}-1.0",
            "count": int(b_high.shape[0]),
            "pct": (b_high.shape[0] / len(s) * 100.0) if len(s) else 0.0,
            "avg_prob": float(b_high.mean()) if len(b_high) else np.nan,
        },
        {
            "model": model_name,
            "bucket": "ALL",
            "count": int(s.shape[0]),
            "pct": 100.0 if len(s) else 0.0,
            "avg_prob": float(s.mean()) if len(s) else np.nan,
        },
    ]
    return pd.DataFrame(rows)


# =========================
# INFERENCE LOOP
# =========================
y_true = []
y_pred_rf = []
y_pred_xgb = []
y_pred_stack = []
y_pred_rule_ml_stack = []

# Probabilitas phishing (versi model label=1)
p_rf = []
p_xgb = []
p_stack = []
p_hybrid = []

errors = []

# Timing decision layer
t_rule_total = 0.0
t_stack_total = 0.0
t_hybrid_decision_total = 0.0

t_all_start = time.perf_counter()

# itertuples lebih cepat dari iterrows untuk data besar
for n, row in enumerate(df.itertuples(index=True), start=1):
    i = row.Index
    url = getattr(row, URL_COL)
    true_label_data = int(getattr(row, LABEL_COL))
    true_label_model = convert_data_label_to_model_label(true_label_data)

    try:
        # 1) Build features
        X_arr = detector.build_ml_vector(url, cols, medians, include_web_content=include_web_content)
        X_df = pd.DataFrame(X_arr, columns=cols)

        # 2) Base models
        rf_raw = int(rf_model.predict(X_df)[0])
        xgb_raw = int(xgb_model.predict(X_df)[0])

        rf_phish_prob = proba_for_label(rf_model, X_df, MODEL_PHISHING_LABEL)
        xgb_phish_prob = proba_for_label(xgb_model, X_df, MODEL_PHISHING_LABEL)

        # 3) Rule timing
        t_rule_0 = time.perf_counter()
        rule_score, rule_category, rule_flag = detector.rule_based_eval(url)  # rule_flag=1 => phishing
        t_rule_total += (time.perf_counter() - t_rule_0)

        # 4) ML + Stacking timing
        t_stack_0 = time.perf_counter()
        if meta_model is not None:
            meta_n_in = int(getattr(meta_model, "n_features_in_", 2))
            meta_feats = [rf_phish_prob, xgb_phish_prob]

            if meta_n_in >= 3:
                meta_feats = [rf_phish_prob, xgb_phish_prob, rule_flag]
            if meta_n_in >= 4:
                meta_feats = [rf_phish_prob, xgb_phish_prob, rule_flag, rule_score]

            meta_X = np.array([meta_feats[:meta_n_in]], dtype=float)
            stack_raw = int(meta_model.predict(meta_X)[0])
            stack_phish_prob = proba_for_label(meta_model, meta_X, MODEL_PHISHING_LABEL)
        else:
            stack_phish_prob = 0.5 * rf_phish_prob + 0.5 * xgb_phish_prob
            stack_raw = MODEL_PHISHING_LABEL if stack_phish_prob >= 0.5 else MODEL_BENIGN_LABEL

        t_stack_total += (time.perf_counter() - t_stack_0)

        # 5) Hybrid timing
        t_hybrid_0 = time.perf_counter()
        hybrid_raw = MODEL_PHISHING_LABEL if rule_flag == 1 else int(stack_raw)
        hybrid_phish_prob = 1.0 if rule_flag == 1 else float(stack_phish_prob)
        t_hybrid_decision_total += (time.perf_counter() - t_hybrid_0)

        # 6) Simpan
        y_true.append(int(true_label_model))
        y_pred_rf.append(int(rf_raw))
        y_pred_xgb.append(int(xgb_raw))
        y_pred_stack.append(int(stack_raw))
        y_pred_rule_ml_stack.append(int(hybrid_raw))

        p_rf.append(clamp01(rf_phish_prob))
        p_xgb.append(clamp01(xgb_phish_prob))
        p_stack.append(clamp01(stack_phish_prob))
        p_hybrid.append(clamp01(hybrid_phish_prob))

    except Exception as e:
        errors.append((i, url, str(e)))

    # Progress per 20k
    if (n % PROGRESS_EVERY == 0) or (n == total_n):
        elapsed = time.perf_counter() - t_all_start
        speed = n / elapsed if elapsed > 0 else 0.0
        remain = total_n - n
        eta_sec = remain / speed if speed > 0 else 0.0

        print(
            f"Progress: {n:,}/{total_n:,} | "
            f"success={len(y_true):,} | error={len(errors):,} | "
            f"speed={speed:,.2f} row/s | ETA={eta_sec/60:,.1f} menit"
        )

t_all = time.perf_counter() - t_all_start

print(f"\nSelesai inferensi. Berhasil: {len(y_true):,} | Gagal: {len(errors):,}")
print(f"Total waktu end-to-end: {t_all:.2f} detik")

if errors:
    err_df = pd.DataFrame(errors[:10], columns=["index", "url", "error"])
    print("Contoh 10 error pertama:")
    display(err_df)


# =========================
# EVALUASI 4 MATRIX
# =========================
evaluate_and_show("Evaluation Matrix - Random Forest", y_true, y_pred_rf)
evaluate_and_show("Evaluation Matrix - XGBoost", y_true, y_pred_xgb)
evaluate_and_show("Evaluation Matrix - ML + Stacking", y_true, y_pred_stack)
evaluate_and_show("Evaluation Matrix - Rule + ML + Stacking", y_true, y_pred_rule_ml_stack)


# =========================
# DISTRIBUSI PROBABILITAS
# Bucket 0.0-0.6 vs 0.6-1.0
# =========================
bucket_rf = bucket_summary("RF", p_rf, split=0.6)
bucket_xgb = bucket_summary("XGB", p_xgb, split=0.6)
bucket_stack = bucket_summary("ML+Stacking", p_stack, split=0.6)
bucket_hybrid = bucket_summary("Rule+ML+Stacking", p_hybrid, split=0.6)

bucket_all = pd.concat([bucket_rf, bucket_xgb, bucket_stack, bucket_hybrid], ignore_index=True)

print("\nDistribusi probabilitas phishing per bucket:")
display(bucket_all)

compare_avg = bucket_all[
    (bucket_all["bucket"] == "ALL")
    & (bucket_all["model"].isin(["ML+Stacking", "Rule+ML+Stacking"]))
][["model", "avg_prob", "count"]]

print("\nPerbandingan rata-rata probabilitas phishing (ALL):")
display(compare_avg)


# =========================
# PERBANDINGAN TIMING
# =========================
n_ok = max(len(y_true), 1)

timing_df = pd.DataFrame(
    [
        {
            "pipeline": "ML+Stacking",
            "total_sec_est": float(t_stack_total),
            "ms_per_row_est": float((t_stack_total / n_ok) * 1000.0),
        },
        {
            "pipeline": "Rule+ML+Stacking",
            "total_sec_est": float(t_rule_total + t_stack_total + t_hybrid_decision_total),
            "ms_per_row_est": float(
                ((t_rule_total + t_stack_total + t_hybrid_decision_total) / n_ok) * 1000.0
            ),
        },
    ]
)

print("\nPerbandingan timing pipeline (decision layer):")
display(timing_df)

print("\nCatatan:")
print("- Evaluasi sudah disamakan ke label model: phishing=1, benign=0.")
print("- Data uji tetap dataset_test.csv.")
print("- Jika include_web_content=True, proses bisa jauh lebih lama karena fetch konten web.")

Total data uji valid: 235,795
Jumlah fitur model: 81
Web content dipakai: True
Meta model tersedia: True
